# Notebook 1: Mô phỏng Học chủ động (Active Learning) trên bộ dữ liệu VietBioNER

Notebook này thực hiện toàn bộ luồng huấn luyện thực nghiệm đối chứng giữa **Nhánh A (Học chủ động đề xuất)** và **Nhánh B (Chọn ngẫu nhiên)**. Toàn bộ mã nguồn được cấu trúc thành các khối mã độc lập chạy tuần tự trên Google Colab.

In [1]:
# Cell 1: Thiết lập môi trường & Cài đặt Thư viện Chuẩn hóa Python 3.12+
import sys
import subprocess
import os

def check_and_install_libs():
    # Danh sách các thư viện cần kiểm tra import (sử dụng torchcrf chữ thường chuẩn)
    required_libs = ['peft', 'seqeval', 'pyvi', 'sentence_transformers', 'torchcrf', 'Levenshtein', 'bitsandbytes', 'triton']
    
    missing_any = False
    for lib in required_libs:
        try:
            __import__(lib)
        except ImportError:
            missing_any = True
            break
            
    if not missing_any:
        print("✓ Môi trường đầy đủ và sẵn sàng cho PEFT/LoRA.")
    else:
        print("✗ Thiếu thư viện hoặc phiên bản không tương thích. Tiến hành dọn dẹp và cài đặt lại phiên bản chuẩn cho Python 3.12...")
        
        # 1. Gỡ cài đặt các bản cũ bị lỗi xung đột binary để làm sạch hoàn toàn môi trường
        print("Đang dọn dẹp các thư viện cũ...")
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", 
                               "bitsandbytes", "peft", "transformers", "accelerate", "triton"])
        
        # 2. Cài đặt các phiên bản tương thích cấu trúc CUDA 12.x và Python 3.12+
        print("Đang tải và cài đặt các phiên bản tương thích cao...")
        libs_to_install = [
            "transformers==4.44.2", 
            "peft==0.12.0", 
            "accelerate==0.34.2", 
            "bitsandbytes==0.43.3", 
            "triton==3.0.0",
            "seqeval==1.2.2", 
            "pyvi==0.1.1", 
            "pytorch-crf==0.7.2", 
            "sentence-transformers==2.7.0",
            "Levenshtein==0.25.1"
        ]
        
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + libs_to_install)
        print("\n Thao tác hoàn tất! Cài đặt thành công phiên bản tối ưu.")
        print("Đang tự động khởi động lại Runtime để áp dụng thay đổi...")
        try:
            os.kill(os.getpid(), 9)
        except Exception:
            pass

check_and_install_libs()


👉 Bỏ qua mount Google Drive (Chi tiết: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.). Có thể đang chạy trên Kaggle hoặc Local.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 64.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.

In [2]:
# Cell 2: Khai báo Thư viện & Cấu hình Hệ thống (Global Configuration)
import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from pyvi import ViTokenizer
from torchcrf import CRF
from seqeval.metrics import f1_score as seq_f1_score, precision_score as seq_prec_score, recall_score as seq_rec_score
import Levenshtein
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
import gc

class Config:
    # Tự động phát hiện môi trường Kaggle vs. Colab/Local
    IS_KAGGLE = os.path.exists("/kaggle")
    
    if IS_KAGGLE:
        INPUT_DIR = "/kaggle/input/datasets/elysia211/active-learning-vietbioner2"
        OUTPUT_DIR = "/kaggle/working"
        DATASET_DIR = os.path.join(INPUT_DIR, "dataset/vietbioner")
        GAZETTEER_DIR = os.path.join(INPUT_DIR, "dataset/gazetteer")
        PREPROCESSED_DIR = os.path.join(OUTPUT_DIR, "dataset/preprocessed")
        LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")
        
        if not os.path.exists(DATASET_DIR) and os.path.exists(INPUT_DIR):
            for root, dirs, files in os.walk(INPUT_DIR):
                if "train.txt" in files or "train.txt".lower() in [f.lower() for f in files]:
                    DATASET_DIR = root
                    break
    else:
        DRIVE_DIR = "/content/drive/MyDrive/active-learning-VietBioNER"
        DATASET_DIR = os.path.join(DRIVE_DIR, "dataset/vietbioner")
        PREPROCESSED_DIR = os.path.join(DRIVE_DIR, "dataset/preprocessed")
        GAZETTEER_DIR = os.path.join(DRIVE_DIR, "dataset/gazetteer")
        LOGS_DIR = os.path.join(DRIVE_DIR, "logs")

    # Siêu tham số mô hình
    MODEL_CHECKPOINT = "manhtt-079/vipubmed-deberta-base"
    MAX_LEN = 256
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    AL_EPOCHS = 12
    PATIENCE = 3
    LAMBDA_LOSS = 1.0

    # Tham số học chủ động
    BUDGET_LIMIT = 0.50  # Chốt giới hạn ngân sách tối đa 50% tập huấn luyện
    BATCH_SELECT = 25    # Mỗi vòng chọn thêm b = 25 câu thô
    THETA = 0.85         # Ngưỡng Cosine Similarity cho bộ lọc Distinct-K
    SEED = 42            # Hạt giống ngẫu nhiên mặc định cho thực nghiệm

    # Danh sách nhãn đích
    LABEL_LIST = ["Symptom_and_Disease", "DiagnosticProcedure", "Location", "DateTime", "Organisation"]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)
os.makedirs(Config.PREPROCESSED_DIR, exist_ok=True)
os.makedirs(Config.LOGS_DIR, exist_ok=True)
print(f"👉 Đường dẫn đọc data: {Config.DATASET_DIR}")
print(f"👉 Đường dẫn lưu file xử lý: {Config.PREPROCESSED_DIR}")
print(f"👉 Đường dẫn lưu logs: {Config.LOGS_DIR}")


👉 Đường dẫn đọc data: /kaggle/input/datasets/elysia211/active-learning-vietbioner2/dataset/vietbioner
👉 Đường dẫn lưu file xử lý: /kaggle/working/dataset/preprocessed
👉 Đường dẫn lưu logs: /kaggle/working/logs


In [14]:
# Cell 1.5: Thiết lập Hugging Face Token
from kaggle_secrets import UserSecretsClient
import os

# Kiểm tra xem đang chạy trên Kaggle hay không
try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ Đã lấy HF_TOKEN từ Kaggle Secrets")
except Exception as e:
    print(f"⚠️ Không thể lấy HF_TOKEN từ Kaggle Secrets: {e}")
    print("👉 Nếu bạn đang chạy local, hãy đặt HF_TOKEN trong biến môi trường hoặc đăng nhập bằng huggingface-cli")

✅ Đã lấy HF_TOKEN từ Kaggle Secrets


In [15]:
# Cập nhật lại Batch Size nhỏ hơn để tránh tràn bộ nhớ
Config.BATCH_SIZE = 8
print(f"Đã cập nhật BATCH_SIZE mới: {Config.BATCH_SIZE}")

Đã cập nhật BATCH_SIZE mới: 8


In [16]:
import torch
import gc

# 1. Cấu hình để tối ưu hóa việc phân mảnh bộ nhớ
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Giải phóng bộ nhớ GPU
def cleanup_gpu():
    if 'model' in globals():
        del globals()['model']
    if 'optimizer' in globals():
        del globals()['optimizer']
    gc.collect()
    torch.cuda.empty_cache()
    print("Đã dọn dẹp bộ nhớ GPU.")


In [17]:
# Cell 3: Tiền xử lý & Tách từ ghép bằng PyVi tự nhiên
def clean_and_normalize_vietnamese(text):
    return text.strip()

def align_segmented_tags(orig_tokens, orig_tags, segmented_tokens):
    new_tags = []
    orig_idx = 0
    for seg_token in segmented_tokens:
        clean_seg = seg_token.replace('_', '').lower()
        accumulated = ""
        tags_in_word = []
        while orig_idx < len(orig_tokens) and len(accumulated) < len(clean_seg):
            accumulated += orig_tokens[orig_idx].lower()
            tags_in_word.append(orig_tags[orig_idx])
            orig_idx += 1

        if not tags_in_word:
            new_tags.append('O')
        else:
            non_o = [t for t in tags_in_word if t != 'O']
            if non_o:
                # ƯU TIÊN: Nếu có nhãn 'B-', ta gán 'B-' để đánh dấu bắt đầu thực thể
                b_tags = [t for t in non_o if t.startswith('B-')]
                if b_tags:
                    new_tags.append(b_tags[0])
                else:
                    new_tags.append(max(set(non_o), key=non_o.count))
            else:
                new_tags.append('O')
    return new_tags

def process_conll_file(file_path, output_path):
    sentences = []
    current_tokens = []
    current_tags = []

    # Kiểm tra an toàn trước khi mở đọc file
    if not os.path.exists(file_path):
        # Sửa lỗi phân biệt chữ Hoa/Thường của Linux trong trường hợp tệp lưu là Train.txt thay vì train.txt
        dir_name = os.path.dirname(file_path)
        base_name = os.path.basename(file_path)
        if os.path.exists(dir_name):
            for real_file in os.listdir(dir_name):
                if real_file.lower() == base_name.lower():
                    file_path = os.path.join(dir_name, real_file)
                    break

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append((current_tokens, current_tags))
                current_tokens, current_tags = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                current_tokens.append(parts[0])
                current_tags.append(parts[1])
        if current_tokens:
            sentences.append((current_tokens, current_tags))

    # Tách từ ghép bằng PyVi tự nhiên và ghi lại file
    with open(output_path, 'w', encoding='utf-8') as out_f:
        for tokens, tags in sentences:
            sentence_text = " ".join(tokens)
            segmented_text = ViTokenizer.tokenize(sentence_text)
            segmented_tokens = segmented_text.split()
            new_tags = align_segmented_tags(tokens, tags, segmented_tokens)

            for tok, tag in zip(segmented_tokens, new_tags):
                out_f.write(f"{tok} {tag}\n")
            out_f.write("\n")
    print(f"Đã tiền xử lý và lưu vào {output_path}")

# Chạy tách từ ghép cho toàn bộ tập dữ liệu huấn luyện chủ động
for split in ['train', 'dev', 'test']:
    raw_path = os.path.join(Config.DATASET_DIR, f"{split}.txt")
    out_path = os.path.join(Config.PREPROCESSED_DIR, f"{split}_segmented.txt")
    process_conll_file(raw_path, out_path)

# Định nghĩa 5 mô tả nhãn tĩnh dạng chuỗi văn bản tự nhiên đã được tách từ ghép bằng PyVi
entity_descriptions = {
    "Symptom_and_Disease": ViTokenizer.tokenize("triệu chứng lâm sàng bệnh lý dấu hiệu bệnh lao phổi"),
    "DiagnosticProcedure": ViTokenizer.tokenize("quy trình chẩn đoán xét nghiệm kỹ thuật y học lâm sàng"),
    "Location": ViTokenizer.tokenize("khu vực địa lý địa điểm vị trí địa danh"),
    "DateTime": ViTokenizer.tokenize("ngày tháng năm khoảng thời gian thời điểm"),
    "Organisation": ViTokenizer.tokenize("tổ chức y tế bệnh viện viện nghiên cứu cơ sở y khoa")
}

with open(os.path.join(Config.PREPROCESSED_DIR, "entity_descriptions_segmented.json"), 'w', encoding='utf-8') as json_f:
    json.dump(entity_descriptions, json_f, ensure_ascii=False, indent=4)
print("Đã lưu tệp entity_descriptions_segmented.json thành công.")

# Ghi tệp metadata.json để tương thích ngược và đánh dấu version thực nghiệm
with open(os.path.join(Config.PREPROCESSED_DIR, "metadata.json"), 'w', encoding='utf-8') as meta_f:
    json.dump({
        "version": "2.0",
        "description": "Mô tả nhãn tĩnh ở dạng văn bản đã tách từ ghép bằng PyVi"
    }, meta_f, ensure_ascii=False, indent=4)
print("Đã lưu tệp metadata.json thành công.")

Đã tiền xử lý và lưu vào /kaggle/working/dataset/preprocessed/train_segmented.txt
Đã tiền xử lý và lưu vào /kaggle/working/dataset/preprocessed/dev_segmented.txt
Đã tiền xử lý và lưu vào /kaggle/working/dataset/preprocessed/test_segmented.txt
Đã lưu tệp entity_descriptions_segmented.json thành công.
Đã lưu tệp metadata.json thành công.


In [21]:
# Cell 4: Thiết lập Lớp Dataset và DataCollator
class VietBioNERDataset(Dataset):
    def __init__(self, data, descriptions, tokenizer, max_len, is_train=True):
        if isinstance(data, str):
            self.sentences = []
            current_tokens = []
            current_tags = []
            with open(data, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        if current_tokens:
                            self.sentences.append((current_tokens, current_tags))
                        current_tokens, current_tags = [], []
                        continue
                    parts = line.split()
                    if len(parts) >= 2:
                        current_tokens.append(parts[0])
                        current_tags.append(parts[1])
                if current_tokens:
                    self.sentences.append((current_tokens, current_tags))
        else: 
            self.sentences = data

        self.descriptions = descriptions
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_list = Config.LABEL_LIST
        self.is_train = is_train

        # Xây dựng các queries
        self.queries = []
        for sent_idx, (tokens, tags) in enumerate(self.sentences):
            present_labels = set()
            for tag in tags:
                if tag != 'O':
                    present_labels.add(tag[2:])
            
            # 1. Giữ lại 100% queries dương tính
            for label_idx, label in enumerate(self.label_list):
                if label in present_labels:
                    self.queries.append((sent_idx, label_idx))
            
            # 2. Negative Query Downsampling (Chỉ trong Train)
            absent_label_indices = [i for i, label in enumerate(self.label_list) if label not in present_labels]
            if self.is_train:
                if absent_label_indices:
                    rand_neg_idx = random.choice(absent_label_indices)
                    self.queries.append((sent_idx, rand_neg_idx))
            else:
                # Pha Validation/Test/Selection: Giữ nguyên đầy đủ cả 5 nhãn truy vấn để có dự đoán trọn vẹn
                for absent_idx in absent_label_indices:
                    self.queries.append((sent_idx, absent_idx))

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        sent_idx, label_idx = self.queries[idx]
        tokens, tags = self.sentences[sent_idx]
        query_label = self.label_list[label_idx]
        query_desc = self.descriptions[query_label]

        binary_tags = []
        for tag in tags:
            if tag != 'O' and tag[2:] == query_label:
                binary_tags.append(tag[0])
            else:
                binary_tags.append('O')

        return {
            "tokens": tokens,
            "tags": binary_tags,
            "query_desc": query_desc,
            "query_label_idx": label_idx
        }

class EntityMaskingCollator:
    def __init__(self, tokenizer, max_len, is_train=True):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        self.tag2id = {'O': 0, 'B': 1, 'I': 2}

    def __call__(self, batch):
        input_ids_batch = []
        attention_mask_batch = []
        token_type_ids_batch = []
        tags_batch = []
        mask_batch = []
        query_label_idx_batch = []

        has_token_type = None

        for item in batch:
            tokens = list(item["tokens"])
            tags = item["tags"]
            query_desc = item["query_desc"]

            # Contextual Word Masking (Chỉ trong Train)
            if self.is_train:
                for i, tag in enumerate(tags):
                    if tag in ['B', 'I']:
                        if random.random() < 0.18:
                            tokens[i] = "[MASK]"
                    else: # tag == 'O'
                        if random.random() < 0.15:
                            tokens[i] = "[MASK]"

            # Mã hóa chuỗi ghép nối: [CLS] s [SEP] d_c [SEP]
            encoding = self.tokenizer(
                tokens,
                query_desc.split(),
                is_split_into_words=True,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

            input_ids = encoding["input_ids"].squeeze(0)
            attention_mask = encoding["attention_mask"].squeeze(0)
            
            if has_token_type is None:
                has_token_type = "token_type_ids" in encoding
            
            if has_token_type:
                token_type_ids = encoding["token_type_ids"].squeeze(0)
                token_type_ids_batch.append(token_type_ids)

            word_ids = encoding.word_ids(batch_index=0)
            sequence_ids = encoding.sequence_ids(0)
            aligned_tags = []
            valid_mask = []

            for t_idx, (w_idx, seq_id) in enumerate(zip(word_ids, sequence_ids)):
                if t_idx == 0:
                    aligned_tags.append(self.tag2id['O'])
                    valid_mask.append(True)
                elif seq_id == 0 and w_idx is not None:
                    tag_str = tags[w_idx]
                    aligned_tags.append(self.tag2id[tag_str])
                    valid_mask.append(True)
                else:
                    aligned_tags.append(self.tag2id['O'])
                    valid_mask.append(False)

            assert len(aligned_tags) == self.max_len, f"Độ dài aligned_tags ({len(aligned_tags)}) không khớp với MAX_LEN ({self.max_len})"

            input_ids_batch.append(input_ids)
            attention_mask_batch.append(attention_mask)
            tags_batch.append(torch.tensor(aligned_tags, dtype=torch.long))
            mask_batch.append(torch.tensor(valid_mask, dtype=torch.bool))
            query_label_idx_batch.append(item["query_label_idx"])

        ret = {
            "input_ids": torch.stack(input_ids_batch),
            "attention_mask": torch.stack(attention_mask_batch),
            "tags": torch.stack(tags_batch),
            "mask": torch.stack(mask_batch),
            "query_label_idx": torch.tensor(query_label_idx_batch, dtype=torch.long)
        }
        if has_token_type:
            ret["token_type_ids"] = torch.stack(token_type_ids_batch)
        return ret

def create_dataloader_from_raw(sentences, descriptions, tokenizer, max_len, is_train=True):
    ds = VietBioNERDataset(sentences, descriptions, tokenizer, max_len, is_train=is_train)
    collator = EntityMaskingCollator(tokenizer, max_len, is_train=is_train)
    return DataLoader(ds, batch_size=Config.BATCH_SIZE, shuffle=is_train, collate_fn=collator)


In [22]:
# Cell 5: Thiết lập Kiến trúc Mô hình ViPubmedDeBERTa-CRF kết hợp LoRA
class MultiTaskBiLSTMCRF(nn.Module):
    def __init__(self, model_checkpoint):
        super(MultiTaskBiLSTMCRF, self).__init__()
        deberta_model = AutoModel.from_pretrained(model_checkpoint)
        
        self.deberta_wrapped = False
        try:
            from peft import LoraConfig, TaskType, get_peft_model
            lora_config = LoraConfig(
                r=8,
                lora_alpha=16,
                lora_dropout=0.1,
                target_modules=["query_proj", "value_proj"],
                bias="none"
            )
            self.deberta = get_peft_model(deberta_model, lora_config)
            self.deberta_wrapped = True
            print("LoRA Adapter successfully integrated into DeBERTa backbone.")
            self.deberta.print_trainable_parameters()
        except Exception as e:
            print(f"PEFT/LoRA integration failed: {e}. Falling back to manual parameter freezing.")
            self.deberta = deberta_model
            if not torch.cuda.is_available():
                print("CPU Mode: Freezing DeBERTa backbone to save RAM.")
                for param in self.deberta.parameters():
                    param.requires_grad = False
        
        hidden_size = deberta_model.config.hidden_size
        self.fc = nn.Linear(hidden_size, 3) # O, B, I
        self.crf = CRF(num_tags=3, batch_first=True)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.deberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True
        )

        last_hidden = outputs.last_hidden_state
        emissions = self.fc(last_hidden)
        cls_embedding = last_hidden[:, 0, :]

        return {
            "emissions": emissions,
            "cls_embedding": cls_embedding
        }


In [23]:
# Cell 6: Thuật toán giải quyết xung đột bằng Xác suất biên CRF Forward-Backward
def compute_marginal_probabilities(emissions, transitions, start_transitions, end_transitions, mask):
    with torch.no_grad():
        emissions = emissions.detach()
        transitions = transitions.detach()
        start_transitions = start_transitions.detach()
        end_transitions = end_transitions.detach()
        mask = mask.detach()

        batch_size, seq_len, num_tags = emissions.shape
        device = emissions.device

        # 1. Forward Pass
        alpha = torch.full((batch_size, seq_len, num_tags), -1e9, device=device)
        alpha[:, 0, :] = start_transitions.unsqueeze(0) + emissions[:, 0, :]
        for t in range(1, seq_len):
            prev_alpha = alpha[:, t-1, :].unsqueeze(2) # (B, K, 1)
            trans = transitions.unsqueeze(0) # (1, K, K)
            next_alpha = torch.logsumexp(prev_alpha + trans, dim=1) + emissions[:, t, :]
            m = mask[:, t].unsqueeze(1)
            alpha[:, t, :] = torch.where(m, next_alpha, alpha[:, t-1, :])

        # 2. Backward Pass (Đã vector hóa hoàn toàn trên GPU)
        beta = torch.full((batch_size, seq_len, num_tags), -1e9, device=device)
        seq_lens = mask.sum(dim=1).long()
        beta[torch.arange(batch_size, device=device), seq_lens - 1, :] = end_transitions.unsqueeze(0)

        for t in range(seq_len - 2, -1, -1):
            next_beta = beta[:, t+1, :].unsqueeze(1) # (B, 1, K)
            trans = transitions.unsqueeze(0) # (1, K, K)
            emit = emissions[:, t+1, :].unsqueeze(1)
            prev_beta = torch.logsumexp(next_beta + trans + emit, dim=2)
            m = mask[:, t+1].unsqueeze(1)
            beta[:, t, :] = torch.where(m, prev_beta, beta[:, t+1, :])

        # 3. Tính xác suất biên (Marginal Probability)
        log_p = alpha + beta
        marginals = torch.softmax(log_p, dim=-1)
        
        # Explicit memory cleanup
        del alpha, beta, log_p
        return marginals

def compute_sentence_entropy(all_emissions, all_masks, transitions, start_transitions, end_transitions):
    # Tối ưu hóa GPU: Tính toán xác suất biên song song bằng batching cả 5 nhãn
    if isinstance(all_emissions, (list, tuple)):
        em = torch.stack(all_emissions) # Shape: (5, seq_len, 3)
    else:
        em = all_emissions

    if isinstance(all_masks, (list, tuple)):
        m = torch.stack(all_masks)      # Shape: (5, seq_len)
    else:
        m = all_masks
    
    marginals = compute_marginal_probabilities(em, transitions, start_transitions, end_transitions, m) # Shape: (5, seq_len, 3)
    
    # Shannon Entropy
    entropy = -torch.sum(marginals * torch.log(marginals + 1e-9), dim=-1) # Shape: (5, seq_len)
    
    total_entropy = 0.0
    for i in range(5):
        entropy_mask = all_masks[i].clone()
        entropy_mask[0] = False
        
        active_entropy = entropy[i, entropy_mask]
        if len(active_entropy) > 0:
            total_entropy += torch.mean(active_entropy).item()
            
    return total_entropy / 5.0

def merge_and_resolve_conflicts(sentence_tokens, all_emissions, all_masks, crf_model, tokenizer, descriptions):
    transitions = crf_model.transitions.detach()
    start_transitions = crf_model.start_transitions.detach()
    end_transitions = crf_model.end_transitions.detach()

    # Tối ưu hóa GPU: Tính xác suất biên song song cho cả 5 nhãn
    if isinstance(all_emissions, (list, tuple)):
        em = torch.stack(all_emissions) # (5, seq_len, 3)
    else:
        em = all_emissions

    if isinstance(all_masks, (list, tuple)):
        m = torch.stack(all_masks)      # (5, seq_len)
    else:
        m = all_masks

    marginals = compute_marginal_probabilities(em, transitions, start_transitions, end_transitions, m) # (5, seq_len, 3)

    # Tối ưu hóa GPU: Giải mã viterbi song song
    viterbi_paths = crf_model.decode(em, mask=m)
    
    label_names = Config.LABEL_LIST
    candidate_spans = []

    for l_idx in range(5):
        encoding = tokenizer(
            sentence_tokens,
            descriptions[label_names[l_idx]].split(),
            is_split_into_words=True,
            max_length=Config.MAX_LEN,
            padding="max_length",
            truncation=True
        )
        word_ids = encoding.word_ids(0)
        sequence_ids = encoding.sequence_ids(0)

        valid_word_ids = []
        for t_idx, (w_idx, seq_id) in enumerate(zip(word_ids, sequence_ids)):
            if t_idx == 0:
                valid_word_ids.append(None)
            elif seq_id == 0 and w_idx is not None and w_idx < len(sentence_tokens):
                valid_word_ids.append(w_idx)
            else:
                valid_word_ids.append(None)

        path = viterbi_paths[l_idx]
        valid_indices = [t for t, w in enumerate(valid_word_ids) if w is not None or t == 0]

        in_entity = False
        start_w = None
        span_probs = []

        for k, t_idx in enumerate(valid_indices):
            if k == 0: continue
            w_idx = valid_word_ids[t_idx]
            tag_id = path[k]
            prob_entity = marginals[l_idx, t_idx, tag_id].item()

            if tag_id == 1: # B-
                if in_entity:
                    prev_t_idx = valid_indices[k-1]
                    prev_w_idx = valid_word_ids[prev_t_idx]
                    if prev_w_idx is not None:
                        candidate_spans.append((start_w, prev_w_idx, l_idx, np.mean(span_probs)))
                start_w = w_idx
                span_probs = [prob_entity]
                in_entity = True
            elif tag_id == 2: # I-
                if in_entity:
                    span_probs.append(prob_entity)
                else:
                    start_w = w_idx
                    span_probs = [prob_entity]
                    in_entity = True
            else: # O
                if in_entity:
                    prev_t_idx = valid_indices[k-1]
                    prev_w_idx = valid_word_ids[prev_t_idx]
                    if prev_w_idx is not None:
                        candidate_spans.append((start_w, prev_w_idx, l_idx, np.mean(span_probs)))
                    in_entity = False
                    span_probs = []
        if in_entity:
            last_t_idx = valid_indices[-1]
            last_w_idx = valid_word_ids[last_t_idx]
            if last_w_idx is not None:
                candidate_spans.append((start_w, last_w_idx, l_idx, np.mean(span_probs)))

    if not candidate_spans:
        return ['O'] * len(sentence_tokens)

    # Giải quyết xung đột span
    candidate_spans = sorted(candidate_spans, key=lambda x: (-x[3], x[1] - x[0], x[2]))
    resolved_spans = []
    occupied_words = set()

    for span in candidate_spans:
        start_w, end_w, l_idx, avg_prob = span
        overlap = any(w in occupied_words for w in range(start_w, end_w + 1))
        if not overlap:
            resolved_spans.append(span)
            occupied_words.update(range(start_w, end_w + 1))

    merged_tags = ['O'] * len(sentence_tokens)
    for start_w, end_w, l_idx, _ in resolved_spans:
        label_name = label_names[l_idx]
        merged_tags[start_w] = 'B-' + label_name
        for w in range(start_w + 1, end_w + 1):
            merged_tags[w] = 'I-' + label_name

    return merged_tags


In [24]:
# Cell 7: Bộ Tăng cường dữ liệu bằng Distant Supervision (DS)
def ds_augment_sentence(tokens, tags, gaz_proc, gaz_org, gaz_date, gaz_loc, gaz_sym):
    spans = []
    current_span = None

    for i, tag in enumerate(tags):
        if tag.startswith('I-') or tag.startswith('B-'):
            etype = tag[2:]
            if etype in ['DiagnosticProcedure', 'Organisation', 'DateTime', 'Location', 'Symptom_and_Disease']:
                if current_span and current_span[2] == etype:
                    current_span = (current_span[0], i, etype)
                else:
                    if current_span:
                        spans.append(current_span)
                    current_span = (i, i, etype)
        else:
            if current_span:
                spans.append(current_span)
            current_span = None
    if current_span:
        spans.append(current_span)

    if not spans:
        return []

    span_to_replace = random.choice(spans)
    start_idx, end_idx, etype = span_to_replace

    # Phân biệt M, Gazetteer và Tỷ lệ kích hoạt động
    if etype == 'DiagnosticProcedure':
        gaz = gaz_proc
        M = 2
        p_activation = 1.0
    elif etype == 'Organisation':
        gaz = gaz_org
        M = 3
        p_activation = 1.0
    elif etype == 'DateTime':
        gaz = gaz_date
        M = 2
        p_activation = 1.0
    elif etype == 'Location':
        gaz = gaz_loc
        M = 2
        p_activation = 1.0
    elif etype == 'Symptom_and_Disease':
        gaz = gaz_sym
        M = 1
        p_activation = 0.50
    else:
        return []

    if not gaz:
        return []

    # Kích hoạt động ngẫu nhiên theo xác suất
    if random.random() > p_activation:
        return []

    chosen_entities = random.sample(gaz, min(M, len(gaz)))

    augmented_sentences = []
    for new_entity_str in chosen_entities:
        segmented_entity = ViTokenizer.tokenize(new_entity_str)
        new_tokens = segmented_entity.split()
        if len(new_tokens) > 0:
            new_tags = ['B-' + etype] + ['I-' + etype] * (len(new_tokens) - 1)
        else:
            new_tags = []

        aug_tokens = tokens[:start_idx] + new_tokens + tokens[end_idx+1:]
        aug_tags = tags[:start_idx] + new_tags + tags[end_idx+1:]
        augmented_sentences.append((aug_tokens, aug_tags))

    return augmented_sentences

def augment_dataset_with_ds(sentences, gaz_proc, gaz_org, gaz_date, gaz_loc, gaz_sym):
    augmented_pool = []
    for tokens, tags in sentences:
        augmented_pool.append((tokens, tags))
        aug_sents = ds_augment_sentence(tokens, tags, gaz_proc, gaz_org, gaz_date, gaz_loc, gaz_sym)
        for aug_tok, aug_tag in aug_sents:
            augmented_pool.append((aug_tok, aug_tag))
    return augmented_pool


In [25]:
# Cell 8: Bộ chọn mẫu Active Learning & Distinct-K Cosine CLS Filter
def select_distinct_k(candidates, predicted_losses, cls_embeddings, b, theta):
    # candidates: List các chỉ số dòng dạng số nguyên (int)
    num_candidates = len(candidates)
    if num_candidates <= b:
        return candidates

    # Tính Cosine Similarity
    norms = np.linalg.norm(cls_embeddings, axis=1, keepdims=True)
    normed_emb = cls_embeddings / (norms + 1e-9)
    cos_sim = np.dot(normed_emb, normed_emb.T)

    # Sắp xếp chỉ số giảm dần
    sorted_indices = np.argsort(predicted_losses)[::-1]

    selected_indices = []
    ignored = set()

    for idx in sorted_indices:
        if idx in ignored:
            continue
        selected_indices.append(idx)
        neighbors = np.where(cos_sim[idx] > theta)[0]
        for neighbor in neighbors:
            ignored.add(neighbor)

    selected_candidates = [candidates[i] for i in selected_indices]

    # Cơ chế fallback Distinct-K Filter động
    # Nếu thiếu câu, nới lỏng theta từ 0.85 lên 0.90 và 0.95
    if len(selected_candidates) < b:
        for fallback_theta in [0.90, 0.95]:
            selected_indices = []
            ignored = set()
            for idx in sorted_indices:
                if idx in ignored:
                    continue
                selected_indices.append(idx)
                neighbors = np.where(cos_sim[idx] > fallback_theta)[0]
                for neighbor in neighbors:
                    ignored.add(neighbor)
            selected_candidates = [candidates[i] for i in selected_indices]
            if len(selected_candidates) >= b:
                break

    # Nếu vẫn thiếu sau khi đã tăng theta lên 0.95, bù bằng các ứng viên có entropy lớn nhất
    if len(selected_candidates) < b:
        selected_set = set(selected_candidates)
        remaining = [c for c in candidates if c not in selected_set]
        # Xây dựng dictionary ánh xạ unique c -> entropy để sort an toàn
        cand_to_entropy = {c: predicted_losses[i] for i, c in enumerate(candidates)}
        sorted_rem = sorted(remaining, key=lambda c: cand_to_entropy[c], reverse=True)
        selected_candidates.extend(sorted_rem[:b - len(selected_candidates)])

    return selected_candidates[:b]


In [26]:
# Cell 9: Simulated Oracle & Token-level Levenshtein Edit Distance
def compute_levenshtein_distance(pred_tags, gold_tags):
    tag_to_char = {}
    char_list_pred = []
    char_list_gold = []

    for tag in pred_tags:
        if tag not in tag_to_char:
            tag_to_char[tag] = chr(len(tag_to_char) + 32)
        char_list_pred.append(tag_to_char[tag])

    for tag in gold_tags:
        if tag not in tag_to_char:
            tag_to_char[tag] = chr(len(tag_to_char) + 32)
        char_list_gold.append(tag_to_char[tag])

    pred_str = "".join(char_list_pred)
    gold_str = "".join(char_list_gold)

    return Levenshtein.distance(pred_str, gold_str)

In [27]:
# Cell 10: Trình quản lý Huấn luyện & Early Stopping
def train_model(model, train_loader, val_loader, epochs, lr, patience, device, is_al_branch=True):
    # Lọc động các tham số thực tế yêu cầu gradient để đưa vào Optimizer (Đảm bảo tương thích CPU/GPU/LoRA)
    optim_groups = []
    
    deberta_params = [p for p in model.deberta.parameters() if p.requires_grad]
    if deberta_params:
        optim_groups.append({"params": deberta_params, "lr": lr})
        
    if hasattr(model, 'bilstm') and model.bilstm is not None:
        bilstm_params = [p for p in model.bilstm.parameters() if p.requires_grad]
        if bilstm_params:
            optim_groups.append({"params": bilstm_params, "lr": 5e-4})
            
    fc_params = [p for p in model.fc.parameters() if p.requires_grad]
    if fc_params:
        optim_groups.append({"params": fc_params, "lr": 5e-4})
        
    crf_params = [p for p in model.crf.parameters() if p.requires_grad]
    if crf_params:
        optim_groups.append({"params": crf_params, "lr": 1e-3})
        
    optimizer = torch.optim.AdamW(optim_groups)
    
    best_val_loss = float('inf')
    patience_counter = 0

    # Sửa đường dẫn temp_checkpoint an toàn trên cả Windows lẫn Linux
    temp_checkpoint = os.path.join(Config.LOGS_DIR, f"temp_best_model_{random.randint(0, 100000)}.pt")
    
    epoch_history = []

    try:
        for epoch in range(epochs):
            model.train()
            total_train_loss = 0

            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                token_type_ids = batch["token_type_ids"].to(device) if "token_type_ids" in batch else None
                tags = batch["tags"].to(device)
                mask = batch["mask"].to(device)

                optimizer.zero_grad()

                outputs = model(input_ids, attention_mask, token_type_ids)
                emissions = outputs["emissions"]

                loss = -model.crf(emissions, tags, mask=mask, reduction='mean')

                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()
                
                del input_ids, attention_mask, token_type_ids, tags, mask, outputs, emissions, loss

            # Pha đánh giá trên tập Validation
            model.eval()
            total_val_loss = 0
            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False):
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    token_type_ids = batch["token_type_ids"].to(device) if "token_type_ids" in batch else None
                    tags = batch["tags"].to(device)
                    mask = batch["mask"].to(device)

                    outputs = model(input_ids, attention_mask, token_type_ids)
                    emissions = outputs["emissions"]
                    ner_loss = -model.crf(emissions, tags, mask=mask, reduction='mean')
                    total_val_loss += ner_loss.item()
                    
                    del input_ids, attention_mask, token_type_ids, tags, mask, outputs, emissions, ner_loss

            avg_val_loss = total_val_loss / len(val_loader)
            avg_train_loss = total_train_loss / len(train_loader)
            print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")
            
            is_best = False
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                torch.save(model.state_dict(), temp_checkpoint)
                is_best = True
            else:
                patience_counter += 1
            
            epoch_history.append({
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "best": is_best
            })
            
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            if patience_counter >= patience:
                print("Dừng sớm (Early Stopping) được kích hoạt!")
                break

        if os.path.exists(temp_checkpoint):
            model.load_state_dict(torch.load(temp_checkpoint))
            
    finally:
        if os.path.exists(temp_checkpoint):
            try:
                os.remove(temp_checkpoint)
            except Exception as e:
                print(f"Cảnh báo: Không thể xóa tệp checkpoint tạm {temp_checkpoint}: {e}")
        
    return {
        "history": epoch_history,
        "best_val_loss": best_val_loss,
        "epoch_stopped": epoch + 1,
        "early_stopped": patience_counter >= patience
    }


In [28]:
# Cell 11: Chạy Huấn luyện Vòng lặp Nhánh A (AL + DS) - CÓ RESUME
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_CHECKPOINT)
from seqeval.metrics import classification_report as seq_class_report

# Thiết lập thư mục log theo SEED để tránh đè dữ liệu
LOGS_SEED_DIR = os.path.join(Config.LOGS_DIR, f"seed_{Config.SEED}")
os.makedirs(LOGS_SEED_DIR, exist_ok=True)

# Đường dẫn file checkpoint và log
PATH_BEST_MODEL = os.path.join(LOGS_SEED_DIR, "best_model_AL.pt")
PATH_LOGS_AL = os.path.join(LOGS_SEED_DIR, "logs_al.json")
PATH_L_INDICES = os.path.join(LOGS_SEED_DIR, "current_L_indices.json")
PATH_SBERT_EMBS = os.path.join(Config.LOGS_DIR, "sbert_embeddings.npy")
PATH_SEED_INDICES = os.path.join(Config.LOGS_DIR, "seed_indices.json")

# Đọc dữ liệu phụ trợ
with open(os.path.join(Config.GAZETTEER_DIR, "diagnostic_procedures_tb.json"), 'r', encoding='utf-8') as f:
    gaz_proc = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "healthcare_organizations.json"), 'r', encoding='utf-8') as f:
    gaz_org = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "datetime.json"), 'r', encoding='utf-8') as f:
    gaz_date = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "location.json"), 'r', encoding='utf-8') as f:
    gaz_loc = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "symptom_and_disease.json"), 'r', encoding='utf-8') as f:
    gaz_sym = json.load(f)
with open(os.path.join(Config.PREPROCESSED_DIR, "entity_descriptions_segmented.json"), 'r', encoding='utf-8') as f:
    descriptions = json.load(f)

def load_raw_conll(file_path):
    sentences = []
    current_tokens = []
    current_tags = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append((current_tokens, current_tags))
                current_tokens, current_tags = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                current_tokens.append(parts[0])
                current_tags.append(parts[1])
        if current_tokens:
            sentences.append((current_tokens, current_tags))
    return sentences

train_segmented = load_raw_conll(os.path.join(Config.PREPROCESSED_DIR, "train_segmented.txt"))
dev_segmented = load_raw_conll(os.path.join(Config.PREPROCESSED_DIR, "dev_segmented.txt"))
test_segmented = load_raw_conll(os.path.join(Config.PREPROCESSED_DIR, "test_segmented.txt"))

# --- LOGIC RESUME AN TOÀN ---
start_loop = 0
logs_al = []
current_L_indices = set()
current_U_indices = set(range(len(train_segmented)))
cumulative_edit_distance = 0
f1 = 0.0

# 1. Đọc indices trước
try:
    if os.path.exists(PATH_L_INDICES):
        with open(PATH_L_INDICES, 'r', encoding='utf-8') as f:
            saved_state = json.load(f)
            current_L_indices = set(saved_state["L_indices"])
            cumulative_edit_distance = int(saved_state.get("cumulative_dist", 0))
        current_U_indices = set(range(len(train_segmented))) - current_L_indices
except (json.JSONDecodeError, FileNotFoundError) as e:
    if isinstance(e, json.JSONDecodeError):
        print(f"Cảnh báo: File current_L_indices.json bị hỏng ({e}). Tạo backup.")
        if os.path.exists(PATH_L_INDICES):
            os.rename(PATH_L_INDICES, PATH_L_INDICES + ".corrupted")
    current_L_indices = set()
    cumulative_edit_distance = 0

# 2. Đọc logs sau
try:
    if os.path.exists(PATH_LOGS_AL):
        with open(PATH_LOGS_AL, 'r', encoding='utf-8') as f:
            logs_al = json.load(f)
        start_loop = len(logs_al)
        if logs_al:
            for item in reversed(logs_al):
                if "overall_f1" in item:
                    f1 = item["overall_f1"]
                    break
                elif "f1" in item:
                    f1 = item["f1"]
                    break
except (json.JSONDecodeError, FileNotFoundError) as e:
    if isinstance(e, json.JSONDecodeError):
        print(f"Cảnh báo: File logs_al.json bị hỏng ({e}). Tạo backup và khởi tạo logs mới.")
        if os.path.exists(PATH_LOGS_AL):
            os.rename(PATH_LOGS_AL, PATH_LOGS_AL + ".corrupted")
    logs_al = []

if current_L_indices and not logs_al:
    start_loop = (len(current_L_indices) - 85) // Config.BATCH_SELECT
    print(f"Khôi phục start_loop = {start_loop} từ tập L_indices hiện tại.")
    for i in range(start_loop):
        logs_al.append({
            "loop": i,
            "seed": Config.SEED,
            "Nt": 85 + i * Config.BATCH_SELECT,
            "Et": 0,
            "round_Et": 0,
            "f1": 0.0,
            "overall_precision": 0.0,
            "overall_recall": 0.0,
            "overall_f1": 0.0,
            "reconstructed": True
        })

# Khởi tạo hoặc nạp S-BERT embeddings
if os.path.exists(PATH_SBERT_EMBS):
    embeddings = np.load(PATH_SBERT_EMBS)
    print(f"Đã nạp thành công S-BERT embeddings pre-computed từ: {PATH_SBERT_EMBS}")
else:
    print("Không tìm thấy tệp S-BERT embeddings. Tiến hành mã hóa...")
    sbert_model = SentenceTransformer('keepitreal/vietnamese-sbert')
    sentence_texts = [" ".join(tokens) for tokens, _ in train_segmented]
    embeddings = sbert_model.encode(sentence_texts, show_progress_bar=True)
    np.save(PATH_SBERT_EMBS, embeddings)
    del sbert_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not current_L_indices:
    print("Bắt đầu phiên huấn luyện mới từ Vòng 0...")
    kmeans = KMeans(n_clusters=85, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    seed_indices = []
    for i in range(85):
        cluster_idx = np.where(kmeans.labels_ == i)[0]
        if len(cluster_idx) == 0: continue
        center = kmeans.cluster_centers_[i]
        dists = np.linalg.norm(embeddings[cluster_idx] - center, axis=1)
        seed_indices.append(int(cluster_idx[np.argmin(dists)]))
    current_L_indices = set(seed_indices)
    current_U_indices = set(range(len(train_segmented))) - current_L_indices
    with open(PATH_SEED_INDICES, 'w', encoding='utf-8') as f_seed:
        json.dump(list(current_L_indices), f_seed)

max_L_size = int(len(train_segmented) * Config.BUDGET_LIMIT)
loop = start_loop

val_loader = create_dataloader_from_raw(dev_segmented, descriptions, tokenizer, Config.MAX_LEN, is_train=False)

while len(current_L_indices) < max_L_size:
    print(f"\n=== Nhánh A (AL) - Vòng lặp {loop} (Labeled size: {len(current_L_indices)}/{max_L_size}) ===")
    L_sentences = [train_segmented[idx] for idx in current_L_indices]
    L_aug = augment_dataset_with_ds(L_sentences, gaz_proc, gaz_org, gaz_date, gaz_loc, gaz_sym)
    
    # --- IN THÔNG TIN VÀ LOG GIAO DIỆN HỌC TẬP --- 
    num_queries_before = len(L_aug) * 5
    temp_ds = VietBioNERDataset(L_aug, descriptions, tokenizer, Config.MAX_LEN, is_train=True)
    num_queries_after = len(temp_ds)
    num_positive, num_negative = 0, 0
    for s_idx, l_idx in temp_ds.queries:
        toks, tgs = temp_ds.sentences[s_idx]
        q_label = temp_ds.label_list[l_idx]
        if any(t != 'O' and t[2:] == q_label for t in tgs):
            num_positive += 1
        else:
            num_negative += 1
            
    entity_counts = {etype: 0 for etype in Config.LABEL_LIST}
    for _, tgs in L_sentences:
        for tg in tgs:
            if tg.startswith('B-'):
                etype = tg[2:]
                if etype in entity_counts:
                    entity_counts[etype] += 1
                    
    print(f"  + Quy mô câu gốc (L_t): {len(L_sentences)} câu")
    print(f"  + Quy mô sau Tăng cường (L_aug): {len(L_aug)} câu (+{len(L_aug) - len(L_sentences)} câu)")
    print(f"  + Queries trước Downsampling: {num_queries_before}")
    print(f"  + Queries sau Downsampling: {num_queries_after} (Tiết kiệm: {((num_queries_before - num_queries_after)/num_queries_before)*100:.1f}%)")
    print(f"    - Queries DƯƠNG TÍNH: {num_positive} | Queries ÂM TÍNH: {num_negative}")
    print("  + Số lượng thực thể gốc được gán nhãn trong pool:")
    for etype, count in entity_counts.items():
        print(f"    - {etype}: {count}")
    print("--------------------------------------------------\n")
    
    train_loader = create_dataloader_from_raw(L_aug, descriptions, tokenizer, Config.MAX_LEN, is_train=True)

    model = MultiTaskBiLSTMCRF(Config.MODEL_CHECKPOINT).to(device)
    train_info = train_model(model, train_loader, val_loader, Config.AL_EPOCHS, Config.LEARNING_RATE, Config.PATIENCE, device, is_al_branch=True)
    epoch_history = train_info["history"]

    torch.save(model.state_dict(), PATH_BEST_MODEL)

    model.eval()
    y_true_all, y_pred_all = [], []
    with torch.no_grad():
        for tokens, tags in tqdm(test_segmented, desc=f'Vòng {loop} - Đánh giá Test', leave=False):
            encodings = tokenizer(
                [tokens] * 5,
                [descriptions[label_name].split() for label_name in Config.LABEL_LIST],
                is_split_into_words=True,
                max_length=Config.MAX_LEN,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = encodings["input_ids"].to(device)
            attention_mask = encodings["attention_mask"].to(device)
            token_type_ids = encodings["token_type_ids"].to(device) if "token_type_ids" in encodings else None
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            
            all_emissions = outputs["emissions"]
            
            all_masks = []
            for i in range(5):
                seq_ids = encodings.sequence_ids(i)
                word_ids = encodings.word_ids(i)
                valid_mask = [
                    (seq_id == 0 and w_idx is not None) or (t_idx == 0)
                    for t_idx, (w_idx, seq_id) in enumerate(zip(word_ids, seq_ids))
                ]
                all_masks.append(torch.tensor(valid_mask, dtype=torch.bool, device=device))
                
            pred = merge_and_resolve_conflicts(tokens, all_emissions, all_masks, model.crf, tokenizer, descriptions)
            assert len(pred) == len(tokens)
            y_pred_all.append(pred)
            y_true_all.append(tags)
            
            del encodings, input_ids, attention_mask, token_type_ids, outputs, all_emissions, all_masks

    prec = seq_prec_score(y_true_all, y_pred_all)
    rec = seq_rec_score(y_true_all, y_pred_all)
    f1 = seq_f1_score(y_true_all, y_pred_all)
    print(f"Vòng {loop} - Test F1-score: {f1:.4f}")
    
    report_dict = seq_class_report(y_true_all, y_pred_all, output_dict=True)
    print("Class-wise Evaluation Report:")
    print(seq_class_report(y_true_all, y_pred_all))
    
    class_wise = {}
    for class_name in Config.LABEL_LIST:
        if class_name in report_dict:
            class_wise[class_name] = {
                "precision": report_dict[class_name]["precision"],
                "recall": report_dict[class_name]["recall"],
                "f1-score": report_dict[class_name]["f1-score"]
            }
        else:
            class_wise[class_name] = {"precision": 0.0, "recall": 0.0, "f1-score": 0.0}

    candidate_indices = list(current_U_indices)
    
    if not candidate_indices:
        print("WARNING: Không còn mẫu nào trong tập U_t để lựa chọn. Dừng vòng lặp active learning.")
        break
        
    candidate_entropies = []
    transitions = model.crf.transitions.detach()
    start_transitions = model.crf.start_transitions.detach()
    end_transitions = model.crf.end_transitions.detach()
    
    with torch.no_grad():
        for idx in tqdm(candidate_indices, desc='Tính toán Entropy cho tập U_t', leave=False):
            tokens, _ = train_segmented[idx]
            encodings = tokenizer(
                [tokens] * 5,
                [descriptions[label_name].split() for label_name in Config.LABEL_LIST],
                is_split_into_words=True,
                max_length=Config.MAX_LEN,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = encodings["input_ids"].to(device)
            attention_mask = encodings["attention_mask"].to(device)
            token_type_ids = encodings["token_type_ids"].to(device) if "token_type_ids" in encodings else None
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            
            all_emissions = outputs["emissions"]
            
            all_masks = []
            for i in range(5):
                seq_ids = encodings.sequence_ids(i)
                word_ids = encodings.word_ids(i)
                valid_mask = [
                    (seq_id == 0 and w_idx is not None) or (t_idx == 0)
                    for t_idx, (w_idx, seq_id) in enumerate(zip(word_ids, seq_ids))
                ]
                all_masks.append(torch.tensor(valid_mask, dtype=torch.bool, device=device))
                
            entropy_score = compute_sentence_entropy(all_emissions, all_masks, transitions, start_transitions, end_transitions)
            candidate_entropies.append(entropy_score)
            
            del encodings, input_ids, attention_mask, token_type_ids, outputs, all_emissions, all_masks
            
        gc.collect()
        torch.cuda.empty_cache()

    candidate_cls = embeddings[candidate_indices]
    batch_selected = select_distinct_k(candidate_indices, candidate_entropies, candidate_cls, Config.BATCH_SELECT, Config.THETA)

    if not batch_selected:
        print("WARNING: Bộ chọn mẫu trả về danh sách rỗng. Dừng vòng lặp.")
        break

    round_edit_distance = 0
    with torch.no_grad():
        for idx in tqdm(batch_selected, desc='Oracle hiệu chỉnh Levenshtein', leave=False):
            tokens, tags = train_segmented[idx]
            encodings = tokenizer(
                [tokens] * 5,
                [descriptions[label_name].split() for label_name in Config.LABEL_LIST],
                is_split_into_words=True,
                max_length=Config.MAX_LEN,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = encodings["input_ids"].to(device)
            attention_mask = encodings["attention_mask"].to(device)
            token_type_ids = encodings["token_type_ids"].to(device) if "token_type_ids" in encodings else None
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            
            all_emissions = outputs["emissions"]
            
            all_masks = []
            for i in range(5):
                seq_ids = encodings.sequence_ids(i)
                word_ids = encodings.word_ids(i)
                valid_mask = [
                    (seq_id == 0 and w_idx is not None) or (t_idx == 0)
                    for t_idx, (w_idx, seq_id) in enumerate(zip(word_ids, seq_ids))
                ]
                all_masks.append(torch.tensor(valid_mask, dtype=torch.bool, device=device))
                
            pred = merge_and_resolve_conflicts(tokens, all_emissions, all_masks, model.crf, tokenizer, descriptions)
            assert len(pred) == len(tokens)
            edit_dist = compute_levenshtein_distance(pred, tags)
            round_edit_distance += int(edit_dist)
            
            del encodings, input_ids, attention_mask, token_type_ids, outputs, all_emissions, all_masks
            
    cumulative_edit_distance += round_edit_distance
    print(f"Vòng {loop} - Chi phí sửa vòng này: {round_edit_distance} - Tổng chi phí sửa lũy kế: {cumulative_edit_distance}")

    logs_al.append({
        "loop": loop, 
        "seed": Config.SEED,
        "Nt": len(current_L_indices), 
        "Et": int(cumulative_edit_distance), 
        "round_Et": int(round_edit_distance), 
        "f1": f1,
        "overall_precision": prec,
        "overall_recall": rec,
        "overall_f1": f1,
        "class_wise": class_wise,
        "epoch_history": epoch_history,
        "best_val_loss": train_info["best_val_loss"],
        "epoch_stopped": train_info["epoch_stopped"],
        "early_stopped": train_info["early_stopped"]
    })

    with open(PATH_LOGS_AL, 'w', encoding='utf-8') as f: json.dump(logs_al, f, indent=4)
    PATH_LOGS_AL_ROOT = os.path.join(Config.LOGS_DIR, "logs_al.json")
    with open(PATH_LOGS_AL_ROOT, 'w', encoding='utf-8') as f: json.dump(logs_al, f, indent=4)

    with open(PATH_L_INDICES, 'w', encoding='utf-8') as f:
        json.dump({"L_indices": list(current_L_indices), "cumulative_dist": int(cumulative_edit_distance)}, f)
    PATH_L_INDICES_ROOT = os.path.join(Config.LOGS_DIR, "current_L_indices.json")
    with open(PATH_L_INDICES_ROOT, 'w', encoding='utf-8') as f:
        json.dump({"L_indices": list(current_L_indices), "cumulative_dist": int(cumulative_edit_distance)}, f)

    current_L_indices.update(batch_selected)
    current_U_indices.difference_update(batch_selected)

    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    loop += 1

if f1 < 0.75:
    print("WARNING: Ngân sách cạn kiệt - Hiệu năng không đạt mục tiêu (F1 < 75%)!")
    logs_al.append({"warning": "Ngân sách cạn kiệt - Hiệu năng không đạt mục tiêu"})
    with open(PATH_LOGS_AL, 'w', encoding='utf-8') as f: json.dump(logs_al, f, indent=4)
    PATH_LOGS_AL_ROOT = os.path.join(Config.LOGS_DIR, "logs_al.json")
    with open(PATH_LOGS_AL_ROOT, 'w', encoding='utf-8') as f: json.dump(logs_al, f, indent=4)

print("Đã chạy xong/tiếp tục Nhánh A.")


Đã nạp thành công S-BERT embeddings pre-computed từ: /kaggle/working/logs/sbert_embeddings.npy
Bắt đầu phiên huấn luyện mới từ Vòng 0...

=== LOG DEBUG TOKENIZATION (KIỂM TRA TẬP ĐẦU VÀO) ===
Câu gốc: ['Các', 'kỹ_thuật', 'nhuộm', 'đặc_biệt', 'còn', 'chưa', 'phổ_biến', 'và', 'không', 'phải', 'bất_cứ', 'bệnh_viện', 'nào', 'cũng', 'làm', 'được', '.']
Mô tả truy vấn: triệu_chứng lâm_sàng bệnh_lý dấu_hiệu bệnh lao_phổi
Word IDs: [None, 0, 1, 2, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Seq IDs: [None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Valid Mask: [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]


=== Nhánh A (AL) - Vòng lặp 0 (Labeled size: 85/409) ===


pytorch_model.bin:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: manhtt-079/vipubmed-deberta-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
deberta.embeddings.position_ids            | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

...

Epoch 9 - Train Loss: 1.4198 - Val Loss: 3.0198
Dừng sớm (Early Stopping) được kích hoạt!


Vòng 2 - Đánh giá Test:   0%|          | 0/171 [00:00<?, ?it/s]

Vòng 2 - Test F1-score: 0.2170
Class-wise Evaluation Report:
                     precision    recall  f1-score   support

           DateTime       0.00      0.00      0.00        32
DiagnosticProcedure       0.21      0.12      0.16        56
           Location       0.00      0.00      0.00        36
       Organisation       0.00      0.00      0.00        18
Symptom_and_Disease       0.34      0.26      0.29       183

          micro avg       0.30      0.17      0.22       325
          macro avg       0.11      0.08      0.09       325
       weighted avg       0.22      0.17      0.19       325



Tính toán Entropy cho tập U_t:   0%|          | 0/1230 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Cell 12: Chạy Huấn luyện Vòng lặp Nhánh B (Random + DS đối chứng) - CÓ RESUME
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_CHECKPOINT)
from seqeval.metrics import classification_report as seq_class_report

# Thiết lập thư mục log theo SEED
LOGS_SEED_DIR = os.path.join(Config.LOGS_DIR, f"seed_{Config.SEED}")
os.makedirs(LOGS_SEED_DIR, exist_ok=True)

# Đường dẫn file checkpoint và log RIÊNG cho Nhánh B
PATH_BEST_MODEL_B = os.path.join(LOGS_SEED_DIR, "best_model_Random.pt")
PATH_LOGS_RANDOM = os.path.join(LOGS_SEED_DIR, "logs_random.json")
PATH_L_INDICES_B = os.path.join(LOGS_SEED_DIR, "current_L_indices_random.json")
PATH_SEED_INDICES = os.path.join(Config.LOGS_DIR, "seed_indices.json")

# Đọc dữ liệu phụ trợ
with open(os.path.join(Config.GAZETTEER_DIR, "diagnostic_procedures_tb.json"), 'r', encoding='utf-8') as f:
    gaz_proc = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "healthcare_organizations.json"), 'r', encoding='utf-8') as f:
    gaz_org = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "datetime.json"), 'r', encoding='utf-8') as f:
    gaz_date = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "location.json"), 'r', encoding='utf-8') as f:
    gaz_loc = json.load(f)
with open(os.path.join(Config.GAZETTEER_DIR, "symptom_and_disease.json"), 'r', encoding='utf-8') as f:
    gaz_sym = json.load(f)
with open(os.path.join(Config.PREPROCESSED_DIR, "entity_descriptions_segmented.json"), 'r', encoding='utf-8') as f:
    descriptions = json.load(f)

def load_raw_conll(file_path):
    sentences = []
    current_tokens = []
    current_tags = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append((current_tokens, current_tags))
                current_tokens, current_tags = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                current_tokens.append(parts[0])
                current_tags.append(parts[1])
        if current_tokens:
            sentences.append((current_tokens, current_tags))
    return sentences

train_segmented = load_raw_conll(os.path.join(Config.PREPROCESSED_DIR, "train_segmented.txt"))
dev_segmented = load_raw_conll(os.path.join(Config.PREPROCESSED_DIR, "dev_segmented.txt"))
test_segmented = load_raw_conll(os.path.join(Config.PREPROCESSED_DIR, "test_segmented.txt"))

# --- LOGIC RESUME NHÁNH B ---
start_loop_b = 0
logs_random = []
current_L_indices = set()
current_U_indices = set(range(len(train_segmented)))
cumulative_edit_distance = 0
f1 = 0.0

# 1. Đọc indices trước
try:
    if os.path.exists(PATH_L_INDICES_B):
        with open(PATH_L_INDICES_B, 'r', encoding='utf-8') as f:
            saved_state = json.load(f)
            current_L_indices = set(saved_state["L_indices"])
            cumulative_edit_distance = int(saved_state.get("cumulative_dist", 0))
        current_U_indices = set(range(len(train_segmented))) - current_L_indices
except (json.JSONDecodeError, FileNotFoundError) as e:
    if isinstance(e, json.JSONDecodeError):
        print(f"Cảnh báo: File current_L_indices_random.json bị hỏng ({e}). Tạo backup.")
        if os.path.exists(PATH_L_INDICES_B):
            os.rename(PATH_L_INDICES_B, PATH_L_INDICES_B + ".corrupted")
    current_L_indices = set()
    cumulative_edit_distance = 0

# 2. Đọc logs sau
try:
    if os.path.exists(PATH_LOGS_RANDOM):
        with open(PATH_LOGS_RANDOM, 'r', encoding='utf-8') as f:
            logs_random = json.load(f)
        start_loop_b = len(logs_random)
        if logs_random:
            for item in reversed(logs_random):
                if "overall_f1" in item:
                    f1 = item["overall_f1"]
                    break
                elif "f1" in item:
                    f1 = item["f1"]
                    break
except (json.JSONDecodeError, FileNotFoundError) as e:
    if isinstance(e, json.JSONDecodeError):
        print(f"Cảnh báo: File logs_random.json bị hỏng ({e}). Tạo backup và khởi tạo logs mới.")
        if os.path.exists(PATH_LOGS_RANDOM):
            os.rename(PATH_LOGS_RANDOM, PATH_LOGS_RANDOM + ".corrupted")
    logs_random = []

if current_L_indices and not logs_random:
    start_loop_b = (len(current_L_indices) - 85) // Config.BATCH_SELECT
    print(f"Nhánh B: Khôi phục start_loop_b = {start_loop_b} từ tập L_indices hiện tại.")
    for i in range(start_loop_b):
        logs_random.append({
            "loop": i,
            "seed": Config.SEED,
            "Nt": 85 + i * Config.BATCH_SELECT,
            "Et": 0,
            "round_Et": 0,
            "f1": 0.0,
            "overall_precision": 0.0,
            "overall_recall": 0.0,
            "overall_f1": 0.0,
            "reconstructed": True
        })

# 3. Phục hồi Seed Set từ Nhánh A hoặc tự chạy KMeans
if not current_L_indices:
    print("Nhánh B: Khởi tạo Seed Set ban đầu...")
    try:
        current_L_indices = set(seed_indices)
    except NameError:
        if os.path.exists(PATH_SEED_INDICES):
            with open(PATH_SEED_INDICES, 'r', encoding='utf-8') as f_state:
                current_L_indices = set(json.load(f_state))
        else:
            print("Cảnh báo: Không tìm thấy Seed Set từ Nhánh A. Tự chạy KMeans...")
            PATH_SBERT_EMBS = os.path.join(Config.LOGS_DIR, "sbert_embeddings.npy")
            if os.path.exists(PATH_SBERT_EMBS):
                embeddings = np.load(PATH_SBERT_EMBS)
            else:
                sbert_model = SentenceTransformer('keepitreal/vietnamese-sbert')
                sentence_texts = [" ".join(tokens) for tokens, _ in train_segmented]
                embeddings = sbert_model.encode(sentence_texts, show_progress_bar=True)
                np.save(PATH_SBERT_EMBS, embeddings)
                del sbert_model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            
            kmeans = KMeans(n_clusters=85, random_state=42, n_init=10)
            kmeans.fit(embeddings)
            seed_indices_local = []
            for i in range(85):
                cluster_idx = np.where(kmeans.labels_ == i)[0]
                if len(cluster_idx) == 0: continue
                center = kmeans.cluster_centers_[i]
                dists = np.linalg.norm(embeddings[cluster_idx] - center, axis=1)
                seed_indices_local.append(int(cluster_idx[np.argmin(dists)]))
            current_L_indices = set(seed_indices_local)
            with open(PATH_SEED_INDICES, 'w', encoding='utf-8') as f_seed:
                json.dump(list(current_L_indices), f_seed)
    current_U_indices = set(range(len(train_segmented))) - current_L_indices

max_L_size = int(len(train_segmented) * Config.BUDGET_LIMIT)
loop = start_loop_b

val_loader = create_dataloader_from_raw(dev_segmented, descriptions, tokenizer, Config.MAX_LEN, is_train=False)

while len(current_L_indices) < max_L_size:
    print(f"\n=== Nhánh B (Random) - Vòng lặp {loop} (Labeled size: {len(current_L_indices)}/{max_L_size}) ===")
    L_sentences = [train_segmented[idx] for idx in current_L_indices]
    L_aug = augment_dataset_with_ds(L_sentences, gaz_proc, gaz_org, gaz_date, gaz_loc, gaz_sym)

    # --- IN THÔNG TIN VÀ LOG GIAO DIỆN HỌC TẬP --- 
    num_queries_before = len(L_aug) * 5
    temp_ds = VietBioNERDataset(L_aug, descriptions, tokenizer, Config.MAX_LEN, is_train=True)
    num_queries_after = len(temp_ds)
    num_positive, num_negative = 0, 0
    for s_idx, l_idx in temp_ds.queries:
        toks, tgs = temp_ds.sentences[s_idx]
        q_label = temp_ds.label_list[l_idx]
        if any(t != 'O' and t[2:] == q_label for t in tgs):
            num_positive += 1
        else:
            num_negative += 1
            
    entity_counts = {etype: 0 for etype in Config.LABEL_LIST}
    for _, tgs in L_sentences:
        for tg in tgs:
            if tg.startswith('B-'):
                etype = tg[2:]
                if etype in entity_counts:
                    entity_counts[etype] += 1
                    
    print(f"  + Quy mô câu gốc (L_t): {len(L_sentences)} câu")
    print(f"  + Quy mô sau Tăng cường (L_aug): {len(L_aug)} câu (+{len(L_aug) - len(L_sentences)} câu)")
    print(f"  + Queries trước Downsampling: {num_queries_before}")
    print(f"  + Queries sau Downsampling: {num_queries_after} (Tiết kiệm: {((num_queries_before - num_queries_after)/num_queries_before)*100:.1f}%)")
    print(f"    - Queries DƯƠNG TÍNH: {num_positive} | Queries ÂM TÍNH: {num_negative}")
    print("  + Số lượng thực thể gốc được gán nhãn trong pool:")
    for etype, count in entity_counts.items():
        print(f"    - {etype}: {count}")
    print("--------------------------------------------------\n")

    train_loader = create_dataloader_from_raw(L_aug, descriptions, tokenizer, Config.MAX_LEN, is_train=True)

    model = MultiTaskBiLSTMCRF(Config.MODEL_CHECKPOINT).to(device)
    train_info = train_model(model, train_loader, val_loader, Config.AL_EPOCHS, Config.LEARNING_RATE, Config.PATIENCE, device, is_al_branch=False)
    epoch_history = train_info["history"]

    torch.save(model.state_dict(), PATH_BEST_MODEL_B)

    model.eval()
    y_true_all, y_pred_all = [], []
    with torch.no_grad():
        for tokens, tags in tqdm(test_segmented, desc=f'Vòng {loop} (B) - Đánh giá Test', leave=False):
            encodings = tokenizer(
                [tokens] * 5,
                [descriptions[label_name].split() for label_name in Config.LABEL_LIST],
                is_split_into_words=True,
                max_length=Config.MAX_LEN,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = encodings["input_ids"].to(device)
            attention_mask = encodings["attention_mask"].to(device)
            token_type_ids = encodings["token_type_ids"].to(device) if "token_type_ids" in encodings else None
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            
            all_emissions = outputs["emissions"]
            
            all_masks = []
            for i in range(5):
                seq_ids = encodings.sequence_ids(i)
                word_ids = encodings.word_ids(i)
                valid_mask = [
                    (seq_id == 0 and w_idx is not None) or (t_idx == 0)
                    for t_idx, (w_idx, seq_id) in enumerate(zip(word_ids, seq_ids))
                ]
                all_masks.append(torch.tensor(valid_mask, dtype=torch.bool, device=device))
                
            pred = merge_and_resolve_conflicts(tokens, all_emissions, all_masks, model.crf, tokenizer, descriptions)
            assert len(pred) == len(tokens)
            y_pred_all.append(pred)
            y_true_all.append(tags)
            
            del encodings, input_ids, attention_mask, token_type_ids, outputs, all_emissions, all_masks

    prec = seq_prec_score(y_true_all, y_pred_all)
    rec = seq_rec_score(y_true_all, y_pred_all)
    f1 = seq_f1_score(y_true_all, y_pred_all)
    print(f"Vòng {loop} (B) - Test F1: {f1:.4f}")
    
    report_dict = seq_class_report(y_true_all, y_pred_all, output_dict=True)
    print("Class-wise Evaluation Report (Nhánh B):")
    print(seq_class_report(y_true_all, y_pred_all))
    
    class_wise = {}
    for class_name in Config.LABEL_LIST:
        if class_name in report_dict:
            class_wise[class_name] = {
                "precision": report_dict[class_name]["precision"],
                "recall": report_dict[class_name]["recall"],
                "f1-score": report_dict[class_name]["f1-score"]
            }
        else:
            class_wise[class_name] = {"precision": 0.0, "recall": 0.0, "f1-score": 0.0}

    candidate_indices = list(current_U_indices)
    
    if not candidate_indices:
        print("WARNING: Không còn ứng viên nào trong tập U_t để chọn ngẫu nhiên. Dừng vòng lặp.")
        break
        
    batch_selected = random.sample(candidate_indices, min(Config.BATCH_SELECT, len(candidate_indices)))

    if not batch_selected:
        print("WARNING: Danh sách chọn mẫu ngẫu nhiên rỗng. Dừng vòng lặp.")
        break

    round_edit_distance = 0
    with torch.no_grad():
        for idx in tqdm(batch_selected, desc='Oracle hiệu chỉnh Levenshtein', leave=False):
            tokens, tags = train_segmented[idx]
            encodings = tokenizer(
                [tokens] * 5,
                [descriptions[label_name].split() for label_name in Config.LABEL_LIST],
                is_split_into_words=True,
                max_length=Config.MAX_LEN,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids = encodings["input_ids"].to(device)
            attention_mask = encodings["attention_mask"].to(device)
            token_type_ids = encodings["token_type_ids"].to(device) if "token_type_ids" in encodings else None
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            
            all_emissions = outputs["emissions"]
            
            all_masks = []
            for i in range(5):
                seq_ids = encodings.sequence_ids(i)
                word_ids = encodings.word_ids(i)
                valid_mask = [
                    (seq_id == 0 and w_idx is not None) or (t_idx == 0)
                    for t_idx, (w_idx, seq_id) in enumerate(zip(word_ids, seq_ids))
                ]
                all_masks.append(torch.tensor(valid_mask, dtype=torch.bool, device=device))
                
            pred = merge_and_resolve_conflicts(tokens, all_emissions, all_masks, model.crf, tokenizer, descriptions)
            assert len(pred) == len(tokens)
            edit_dist = compute_levenshtein_distance(pred, tags)
            round_edit_distance += int(edit_dist)
            
            del encodings, input_ids, attention_mask, token_type_ids, outputs, all_emissions, all_masks
            
    cumulative_edit_distance += round_edit_distance
    print(f"Vòng {loop} (B) - Chi phí sửa vòng này: {round_edit_distance} - Tổng chi phí sửa lũy kế: {cumulative_edit_distance}")

    logs_random.append({
        "loop": loop, 
        "seed": Config.SEED,
        "Nt": len(current_L_indices), 
        "Et": int(cumulative_edit_distance), 
        "round_Et": int(round_edit_distance), 
        "f1": f1,
        "overall_precision": prec,
        "overall_recall": rec,
        "overall_f1": f1,
        "class_wise": class_wise,
        "epoch_history": epoch_history,
        "best_val_loss": train_info["best_val_loss"],
        "epoch_stopped": train_info["epoch_stopped"],
        "early_stopped": train_info["early_stopped"]
    })

    with open(PATH_LOGS_RANDOM, 'w', encoding='utf-8') as f: json.dump(logs_random, f, indent=4)
    PATH_LOGS_RANDOM_ROOT = os.path.join(Config.LOGS_DIR, "logs_random.json")
    with open(PATH_LOGS_RANDOM_ROOT, 'w', encoding='utf-8') as f: json.dump(logs_random, f, indent=4)

    with open(PATH_L_INDICES_B, 'w', encoding='utf-8') as f:
        json.dump({"L_indices": list(current_L_indices), "cumulative_dist": int(cumulative_edit_distance)}, f)
    PATH_L_INDICES_B_ROOT = os.path.join(Config.LOGS_DIR, "current_L_indices_random.json")
    with open(PATH_L_INDICES_B_ROOT, 'w', encoding='utf-8') as f:
        json.dump({"L_indices": list(current_L_indices), "cumulative_dist": int(cumulative_edit_distance)}, f)

    current_L_indices.update(batch_selected)
    current_U_indices.difference_update(batch_selected)

    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    loop += 1

if f1 < 0.75:
    print("WARNING (Nhánh B): Ngân sách cạn kiệt - Hiệu năng không đạt mục tiêu (F1 < 75%)!")
    logs_random.append({"warning": "Ngân sách cạn kiệt - Hiệu năng không đạt mục tiêu"})
    with open(PATH_LOGS_RANDOM, 'w', encoding='utf-8') as f: json.dump(logs_random, f, indent=4)
    PATH_LOGS_RANDOM_ROOT = os.path.join(Config.LOGS_DIR, "logs_random.json")
    with open(PATH_LOGS_RANDOM_ROOT, 'w', encoding='utf-8') as f: json.dump(logs_random, f, indent=4)

print("Đã hoàn thành/tiếp tục Nhánh B (Random).")
